# Final annotation run

Qwen3-VL-32B via OpenRouter. Setup and functions first, then a ten-article check, then the eight outlets.

## Setup

In [ ]:
# imports
import json
import os
import re
import time
import base64
import io
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from PIL import Image as PILImage
from openai import OpenAI, RateLimitError, APIError, APIConnectionError

In [ ]:
# openrouter client, the key is read from the environment and never written in this file
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["openrouter_api_key"])
model_name = "qwen/qwen3-vl-32b-instruct"

In [ ]:
# paths, prompt files and run settings
final_dir = Path(".")
codebook_text = Path("codebook_updated21.8.txt").read_text(encoding="utf-8")
instruction_text = Path("instruction_text.txt").read_text(encoding="utf-8")

max_image_dimension = 1024
jpeg_quality = 85
max_retries = 5
retry_base_delay = 20
max_workers = 10
checkpoint_every = 100

## Sending an article to the model

In [ ]:
# encode a local image file as a base64 data uri
def encode_image(path):
    with PILImage.open(path) as img:
        img = img.convert("RGB")
        img.thumbnail((max_image_dimension, max_image_dimension), PILImage.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=jpeg_quality)
        encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


# the image files listed for one article
def list_images(row):
    if pd.isna(row["image_filename"]):
        return []
    return [name.strip() for name in str(row["image_filename"]).split(";") if name.strip()]


# the listed image files that are actually present on disk
def get_image_paths(row):
    paths = [Path(row["images_folder"]) / name for name in list_images(row)]
    return [path for path in paths if path.exists()]

In [ ]:
# build the request payload: manual and instructions, the headline, the text, then each image
def build_messages(title, article_text, image_paths):
    content = [
        {"type": "text", "text": codebook_text + "\n\n" + instruction_text,
         "cache_control": {"type": "ephemeral"}},
        {"type": "text", "text": "Article headline:\n" + str(title)},
        {"type": "text", "text": "Article text:\n" + str(article_text)},
    ]
    for path in image_paths:
        content.append({"type": "text", "text": f"Image file name: {path.name}"})
        content.append({"type": "image_url", "image_url": {"url": encode_image(path)}})
    return [{"role": "user", "content": content}]


# send one request, retrying network and rate-limit errors with increasing delay
def call_with_retries(**kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            return client.chat.completions.create(**kwargs)
        except (RateLimitError, APIError, APIConnectionError):
            if attempt == max_retries:
                raise
            time.sleep(retry_base_delay * attempt)


# code one article and return the parsed object with usage
def code_article(row):
    start = time.time()
    response = call_with_retries(
        model=model_name,
        messages=build_messages(row["title"], row["text"], get_image_paths(row)),
        temperature=0, top_p=1, seed=42, frequency_penalty=0, presence_penalty=0,
        response_format={"type": "json_object"},
    )
    try:
        parsed = json.loads(response.choices[0].message.content)
    except (json.JSONDecodeError, TypeError):
        parsed = None
    return {
        "parsed": parsed,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "cost": getattr(response.usage, "cost", None),
        "seconds": time.time() - start,
    }

## Running an outlet

In [ ]:
# read a results file without pandas turning empty strings into nan
def read_results_csv(path):
    return pd.read_csv(path, keep_default_na=False, na_values=[], low_memory=False)


# parse the stored json of one result row, returning none when it is empty, malformed, or lists categories without their fields
def parse_row(value):
    try:
        parsed = json.loads(value) if value else None
    except (json.JSONDecodeError, TypeError):
        return None
    if isinstance(parsed, dict) and not all(isinstance(e, dict) for e in parsed.get("activated_categories") or []):
        return None
    return parsed


# true when a checkpoint row holds a usable result, false when the article must be sent again
def is_usable(row):
    parsed = parse_row(row.get("parsed"))
    return isinstance(parsed, dict) and "activated_categories" in parsed


# count the activations held in one stage column of a results frame
def count_activations(results_df, column="parsed"):
    rows = results_df[column].apply(parse_row)
    return sum(len(p["activated_categories"]) for p in rows
               if isinstance(p, dict) and "activated_categories" in p)


# the document the model was given and that evidence is checked against
def coded_document(title, text):
    return str(title) + "\n" + str(text)

In [ ]:
# code every article of one outlet, resuming from the checkpoint and retrying unusable rows
# code every article of one outlet, resuming from the checkpoint and retrying unusable rows
def run_outlet(source_df, checkpoint_path):
    valid_ids = set(source_df["article_url"])
    done_rows = {}

    if checkpoint_path.exists():
        existing = read_results_csv(checkpoint_path)
        for _, row in existing.iterrows():
            record = row.to_dict()
            if record["article_id"] in valid_ids and is_usable(record):
                done_rows[record["article_id"]] = record
        print(f"  resuming from {len(done_rows)} usable rows")

    remaining_df = source_df[~source_df["article_url"].isin(done_rows)]
    total = len(source_df)
    done = len(done_rows)
    failed = []
    print(f"  sending {len(remaining_df)} of {total} articles on {max_workers} workers")

    def save():
        pd.DataFrame(list(done_rows.values())).to_csv(
            checkpoint_path, index=False, encoding="utf-8-sig")

    executor = ThreadPoolExecutor(max_workers=max_workers)
    futures = {executor.submit(code_article, row): row for _, row in remaining_df.iterrows()}
    try:
        for future in as_completed(futures):
            row = futures[future]
            try:
                result = future.result()
            except Exception as exc:
                failed.append({"article_id": row["article_url"], "error": str(exc)})
                continue
            done_rows[row["article_url"]] = {
                "model": model_name,
                "article_id": row["article_url"],
                "title": row["title"],
                "parsed": json.dumps(result["parsed"], ensure_ascii=False) if result["parsed"] else "",
                "prompt_tokens": result["prompt_tokens"],
                "completion_tokens": result["completion_tokens"],
                "cost": result["cost"],
                "seconds": result["seconds"],
            }
            done += 1
            if done % checkpoint_every == 0:
                save()
                print(f"  {done}/{total}")
    except KeyboardInterrupt:
        print(f"  interrupted at {done}/{total}")
    finally:
        executor.shutdown(wait=False, cancel_futures=True)
        save()

    if failed:
        pd.DataFrame(failed).to_csv(final_dir / f"{checkpoint_path.stem}_failed.csv",
                                    index=False, encoding="utf-8-sig")
        print(f"  {len(failed)} articles failed permanently")
    return pd.DataFrame(list(done_rows.values()))

## The three filters of section 4.5

In [ ]:
# named category labels the model returns without their number, mapped by orientation
legit_map = {
    "Authorization": "6.1 Authorization",
    "Moral Evaluation": "6.2 Moral Evaluation",
    "Rationalization": "6.3 Rationalization",
    "Mythopoesis": "6.4 Mythopoesis",
    "Altruism": "6.5 Altruism",
    "Emotion": "6.6 Emotion",
    "Victimization": "6.7 Victimization",
}
delegit_map = {
    "Authorization": "7.1 Authorization",
    "Moral Evaluation": "7.2 Moral Evaluation",
    "Rationalization": "7.3 Rationalization",
    "Mythopoesis": "7.4 Mythopoesis",
    "Emotion": "7.5 Emotion",
    "Demonization and Dehumanization": "7.6 Demonization and Dehumanization",
    "Outcasting": "7.7 Outcasting",
}

# role russia must occupy for each category, following section 4.5
required_role = {
    "6.1 Authorization": {"target"},
    "6.2 Moral Evaluation": {"agent"},
    "6.3 Rationalization": {"agent"},
    "6.4 Mythopoesis": {"agent", "topic"},
    "6.5 Altruism": {"provider"},
    "6.6 Emotion": {"topic"},
    "6.7 Victimization": {"target"},
    "7.1 Authorization": {"target"},
    "7.2 Moral Evaluation": {"agent"},
    "7.3 Rationalization": {"agent"},
    "7.4 Mythopoesis": {"agent"},
    "7.5 Emotion": {"agent"},
    "7.6 Demonization and Dehumanization": {"target"},
    "7.7 Outcasting": {"target"},
}


# map a named category label onto its numbered form, returning none when there is no match
def canonical_label(category, orientation):
    category = (category or "").strip()
    if re.match(r"^\d", category):
        return category
    return {"legitimation": legit_map,
            "delegitimation": delegit_map}.get(orientation, {}).get(category)

In [ ]:
# first filter: replace a named category label with its numbered form, drop labels with no match
def apply_category_canonicalization(results_df, column):
    relabeled, dropped = [], []
    new_parsed = []

    for _, row in results_df.iterrows():
        parsed = parse_row(row[column])
        if not isinstance(parsed, dict) or "activated_categories" not in parsed:
            new_parsed.append(row[column])
            continue

        kept = []
        for entry in parsed["activated_categories"]:
            category = (entry.get("category") or "").strip()
            orientation = entry.get("orientation")
            label = canonical_label(category, orientation)
            if label is None:
                dropped.append({"article_id": row["article_id"], "category": category,
                                "orientation": orientation})
                continue
            if label != category:
                relabeled.append({"article_id": row["article_id"], "original_category": category,
                                  "orientation": orientation, "canonical_category": label})
                entry = dict(entry)
                entry["category"] = label
            kept.append(entry)

        parsed["activated_categories"] = kept
        new_parsed.append(json.dumps(parsed, ensure_ascii=False))

    return new_parsed, {"relabeled": relabeled, "dropped_unknown": dropped}

In [ ]:
# normalize whitespace and spacing before punctuation before comparing text
def normalize_for_grounding(text):
    text = re.sub(r"\s+", " ", text).strip()
    return re.sub(r"\s+([.,!?;:])", r"\1", text)


# check that the evidence is verbatim: empty, one exact span, or spans joined by an ellipsis
def is_grounded(text_evidence, article_text):
    if not text_evidence:
        return True
    article = normalize_for_grounding(str(article_text))
    fragments = [f.strip() for f in text_evidence.replace("\u2026", "...").split("...") if f.strip()]
    if not fragments:
        return False
    search_from = 0
    for fragment in fragments:
        fragment = normalize_for_grounding(fragment)
        position = article.find(fragment, search_from)
        if position == -1:
            return False
        search_from = position + len(fragment)
    return True


# second filter: require the quoted evidence to occur in the article word for word
# when the quotation is not found, the text evidence is removed, and an activation that also
# rests on an image is kept as an image activation
def apply_grounding_filter(results_df, source_df, column):
    article_text = {row["article_url"]: coded_document(row["title"], row["text"])
                    for _, row in source_df.iterrows()}
    dropped = []
    new_parsed = []

    for _, row in results_df.iterrows():
        parsed = parse_row(row[column])
        if not isinstance(parsed, dict) or "activated_categories" not in parsed:
            new_parsed.append(row[column])
            continue

        kept = []
        for entry in parsed["activated_categories"]:
            evidence = entry.get("text_evidence")
            if is_grounded(evidence, article_text.get(row["article_id"], "")):
                kept.append(entry)
                continue
            image_kept = bool(entry.get("image_evidence"))
            dropped.append({"article_id": row["article_id"], "category": entry.get("category"),
                            "orientation": entry.get("orientation"), "text_evidence": evidence,
                            "kept_as_image": image_kept})
            if image_kept:
                entry = dict(entry)
                entry["text_evidence"] = None
                kept.append(entry)

        parsed["activated_categories"] = kept
        new_parsed.append(json.dumps(parsed, ensure_ascii=False))

    return new_parsed, {"dropped_grounding": dropped}

In [ ]:
# third filter: require the role returned to match the role the category requires
def apply_role_filter(results_df, column):
    dropped = []
    new_parsed = []

    for _, row in results_df.iterrows():
        parsed = parse_row(row[column])
        if not isinstance(parsed, dict) or "activated_categories" not in parsed:
            new_parsed.append(row[column])
            continue

        kept = []
        for entry in parsed["activated_categories"]:
            allowed = required_role.get(entry.get("category"))
            if allowed and entry.get("russia_role") in allowed:
                kept.append(entry)
                continue
            dropped.append({"article_id": row["article_id"], "category": entry.get("category"),
                            "orientation": entry.get("orientation"),
                            "russia_role": entry.get("russia_role"),
                            "expected_role": sorted(allowed) if allowed else None,
                            "text_evidence": entry.get("text_evidence")})

        parsed["activated_categories"] = kept
        new_parsed.append(json.dumps(parsed, ensure_ascii=False))

    return new_parsed, {"dropped_role": dropped}


# run the three filters of section 4.5 in order, keeping each stage in its own column
# parsed            what the model returned, untouched
# parsed_canonical  after category labels were canonicalized
# parsed_grounded   after the grounding filter
# parsed_final      after the role filter as well
def apply_filters(results_df, source_df):
    out = results_df.copy()
    out["parsed_canonical"], logs = apply_category_canonicalization(out, "parsed")
    out["parsed_grounded"], grounding_logs = apply_grounding_filter(out, source_df, "parsed_canonical")
    out["parsed_final"], role_logs = apply_role_filter(out, "parsed_grounded")
    logs.update(grounding_logs)
    logs.update(role_logs)
    return out, logs


# print how many activations survive each step, the three shares section 4.5 reports
def report_filter_shares(filtered_df, logs):
    stages = [("returned", "parsed"), ("after canonicalization", "parsed_canonical"),
              ("after grounding", "parsed_grounded"), ("after role", "parsed_final")]
    counts = [(label, count_activations(filtered_df, column)) for label, column in stages]
    returned = counts[0][1]
    for label, value in counts:
        print(f"  activations {label:22s} {value:7d}")
    print(f"  relabeled {len(logs['relabeled'])}")
    for name in ["dropped_unknown", "dropped_grounding", "dropped_role"]:
        print(f"    {name:18s} {len(logs[name]):7d} "
              f"({len(logs[name]) / max(returned, 1) * 100:5.2f}% of returned)")

## Output

In [ ]:
# print one article, what the model received for it, and what it returned
def show_article(row, result):
    listed = list_images(row)
    found = get_image_paths(row)
    print("=" * 96)
    print(row["title"])
    print(row["article_url"])
    print(f"  sent: headline + {len(str(row['text']))} chars, {len(found)} of {len(listed)} images, "
          f"{result['prompt_tokens']} prompt tokens")

    parsed = result["parsed"]
    if parsed is None:
        print("  model returned output that could not be parsed")
        return

    entries = parsed.get("activated_categories") or []
    if not entries:
        print("  no categories activated")
    for entry in entries:
        label = canonical_label(entry.get("category"), entry.get("orientation"))
        allowed = required_role.get(label)
        if label is None:
            verdict = "would be dropped, unknown category"
        elif entry.get("russia_role") not in allowed:
            verdict = f"would be dropped, role, expected {sorted(allowed)}"
        elif not is_grounded(entry.get("text_evidence"), coded_document(row["title"], row["text"])):
            verdict = "would be dropped, evidence not found in the article"
        else:
            verdict = "kept"
        print(f"  {label or entry.get('category')} | {entry.get('orientation')} | "
              f"russia as {entry.get('russia_role')} | {verdict}")
        if entry.get("text_evidence"):
            print(f"      text: {entry['text_evidence']}")
        for image in entry.get("image_evidence") or []:
            print(f"      image: {image.get('image_file')} | {image.get('reason')}")

    print(f"  reasoning: {parsed.get('reasoning')}")

In [ ]:
# print what the model produced across one outlet
def summarize(source, filtered_df):
    rows = filtered_df["parsed_final"].apply(parse_row)
    coded = rows.dropna()
    n = max(len(coded), 1)
    category_sets = coded.apply(lambda p: {e["category"] for e in p["activated_categories"]})
    per_article = Counter(category for s in category_sets for category in s)
    per_activation = Counter(e["category"] for p in coded for e in p["activated_categories"])
    empty = int((category_sets.apply(len) == 0).sum())

    print(f"\n{source}")
    print(f"  articles coded {len(coded)}, unusable {int(rows.isna().sum())}, "
          f"without any activation {empty} ({empty / n * 100:.1f}%)")
    print()
    table = pd.DataFrame([{"category": category,
                           "articles": per_article[category],
                           "share": f"{per_article[category] / n * 100:.1f}%",
                           "activations": per_activation[category]}
                          for category in sorted(required_role)])
    print(table.to_string(index=False))
    cost = pd.to_numeric(filtered_df["cost"], errors="coerce").sum()
    seconds = pd.to_numeric(filtered_df["seconds"], errors="coerce").sum()
    print(f"\n  cost ${cost:.2f}, model time {seconds / 3600:.1f} h")

In [ ]:
# load one outlet, code it, apply the three filters, save the results and the logs
def process_outlet(source):
    print(source)
    source_df = read_results_csv(final_dir / f"{source}_final_data_v2.csv")
    source_df["images_folder"] = str(final_dir / f"{source}_final_images")
    print(f"  {len(source_df)} articles")

    results_df = run_outlet(source_df, final_dir / f"{source}_checkpoint.csv")
    filtered_df, logs = apply_filters(results_df, source_df)

    filtered_df.to_csv(final_dir / f"{source}_results_final.csv", index=False, encoding="utf-8-sig")
    for name, log_rows in logs.items():
        pd.DataFrame(log_rows).to_csv(final_dir / f"{source}_log_{name}.csv",
                                      index=False, encoding="utf-8-sig")

    report_filter_shares(filtered_df, logs)
    summarize(source, filtered_df)
    return filtered_df

## Check on ten articles

Ten articles through the model, printed in full, nothing saved.

In [ ]:
# send ten articles and show what the model received and returned for each
def smoke_test(source, n=10):
    source_df = read_results_csv(final_dir / f"{source}_final_data_v2.csv")
    source_df["images_folder"] = str(final_dir / f"{source}_final_images")
    for _, row in source_df.sample(n=n, random_state=42).iterrows():
        show_article(row, code_article(row))

In [ ]:
smoke_test("rp")

## Germany

### Der Spiegel

In [ ]:
# check the spiegel file and any old checkpoint before the full run
df = read_results_csv(final_dir / "spiegel_final_data_v2.csv")
df["images_folder"] = str(final_dir / "spiegel_final_images")

print("articles:", len(df))
print("title over 300 chars:", int((df["title"].str.len() > 300).sum()))
print("empty text:", int((df["text"].str.strip() == "").sum()))
print("text starting with [:", int(df["text"].str.startswith("[").sum()))
print("median text chars:", int(df["text"].str.len().median()))
print("articles with all images on disk:",
      int(df.apply(lambda r: len(get_image_paths(r)) == len(list_images(r)), axis=1).sum()))
print("old checkpoint exists:", (final_dir / "spiegel_checkpoint.csv").exists())

In [ ]:
spiegel_results = process_outlet("spiegel")

### Die Welt

In [ ]:
welt_results = process_outlet("welt")

In [ ]:
# compare paid welt fragments with full free articles
def n_categories(value):
    parsed = parse_row(value)
    return len({e["category"] for e in parsed["activated_categories"]}) if parsed else None

welt_results["paid"] = welt_results["article_id"].str.contains("/plus", na=False)
welt_results["n_final"] = welt_results["parsed_final"].apply(n_categories)
print(welt_results.groupby("paid")["n_final"].agg(
    articles="size",
    no_activation=lambda s: f"{(s == 0).mean() * 100:.1f}%",
    mean_categories="mean"))

In [ ]:
# legitimating share among activations, paid fragments against full articles
def counts(value):
    parsed = parse_row(value)
    if not parsed:
        return (0, 0)
    cats = {(e["category"], e["orientation"]) for e in parsed["activated_categories"]}
    return (sum(o == "legitimation" for _, o in cats), len(cats))

welt_results[["legit", "total"]] = welt_results["parsed_final"].apply(counts).tolist()
share = welt_results.groupby("paid")[["legit", "total"]].sum()
share["legitimating share"] = (share["legit"] / share["total"] * 100).round(1)
print(share)

## Spain

### elDiario.es

In [ ]:
# strip the membership block from eldiario bodies, drop articles left below the length threshold, and save under the same name
import shutil
import pandas as pd

shutil.copy("eldiario_final_data_v2.csv", "eldiario_final_data_v2_backup.csv")

def strip_membership_block(text):
    marker = "no causar baja como socio/a de elDiario.es"
    position = str(text).find(marker)
    return str(text)[position + len(marker):].lstrip() if 0 <= position < 600 else str(text)

df = pd.read_csv("eldiario_final_data_v2.csv", keep_default_na=False, na_values=[], low_memory=False)
before = df["text"].str.len()
df["text"] = df["text"].apply(strip_membership_block)
stripped = df["text"].str.len() < before
too_short = stripped & (df["text"].str.len() < 268)

print("stripped:", int(stripped.sum()))
print("dropped:", int(too_short.sum()))
df = df[~too_short]
print("left:", len(df))
df.to_csv("eldiario_final_data_v2.csv", index=False, encoding="utf-8-sig")

In [ ]:
eldiario_results = process_outlet("eldiario")

In [ ]:
# share of articles without any activation within length bands, per outlet
rows = []
for source in ["eldiario", "rp", "welt"]:
    results = read_results_csv(final_dir / f"{source}_results_final.csv")
    data = read_results_csv(final_dir / f"{source}_final_data_v2.csv")[["article_url", "text"]]
    merged = results.merge(data, left_on="article_id", right_on="article_url")
    parsed = merged["parsed_final"].apply(parse_row)
    merged = merged[parsed.apply(lambda p: isinstance(p, dict) and "activated_categories" in p)]
    merged["empty"] = merged["parsed_final"].apply(lambda s: len(parse_row(s)["activated_categories"]) == 0)
    merged["band"] = pd.cut(merged["text"].str.len(), [0, 1000, 2000, 4000, 10**7],
                            labels=["<1000", "1000-2000", "2000-4000", ">4000"])
    merged["outlet"] = source
    rows.append(merged[["outlet", "band", "empty"]])

table = pd.concat(rows).groupby(["band", "outlet"], observed=True)["empty"].agg(["mean", "size"])
table["mean"] = (table["mean"] * 100).round(1)
print(table.unstack("outlet").to_string())

### El Mundo

### old data without title

In [ ]:
# apply the three filters of section 4.5 to the el mundo run made with the old notebook
# grounding uses the text alone, because that run did not send the headline
results = read_results_csv(final_dir / "elmundo_full_results_raw.csv")
source = read_results_csv(final_dir / "elmundo_final_data.csv")
document = dict(zip(source["article_url"], source["text"]))

results = results[results["article_id"].isin(document)].reset_index(drop=True)
print(f"{len(results)} rows kept after dropping ids that are not in the source file")

logs = {"relabeled": [], "dropped_unknown": [], "dropped_grounding": [], "dropped_role": []}
canonical_column, grounded_column, final_column = [], [], []

for _, row in results.iterrows():
    parsed = parse_row(row["parsed"])
    if not isinstance(parsed, dict) or "activated_categories" not in parsed:
        canonical_column.append(row["parsed"])
        grounded_column.append(row["parsed"])
        final_column.append(row["parsed"])
        continue

    article_id = row["article_id"]
    text = document[article_id]
    canonical, grounded, final = [], [], []

    for entry in parsed["activated_categories"]:
        category = (entry.get("category") or "").strip()
        orientation = entry.get("orientation")
        label = canonical_label(category, orientation)

        if label is None or label not in required_role:
            logs["dropped_unknown"].append({"article_id": article_id, "category": category,
                                            "orientation": orientation})
            continue
        if label != category:
            logs["relabeled"].append({"article_id": article_id, "original_category": category,
                                      "orientation": orientation, "canonical_category": label})
            entry = dict(entry)
            entry["category"] = label
        canonical.append(entry)

        if not is_grounded(entry.get("text_evidence"), text):
            logs["dropped_grounding"].append({"article_id": article_id, "category": label,
                                              "orientation": orientation,
                                              "text_evidence": entry.get("text_evidence")})
            continue
        grounded.append(entry)

        allowed = required_role[label]
        if entry.get("russia_role") not in allowed:
            logs["dropped_role"].append({"article_id": article_id, "category": label,
                                         "orientation": orientation,
                                         "russia_role": entry.get("russia_role"),
                                         "expected_role": sorted(allowed),
                                         "text_evidence": entry.get("text_evidence")})
            continue
        final.append(entry)

    for column, kept in [(canonical_column, canonical), (grounded_column, grounded),
                         (final_column, final)]:
        column.append(json.dumps({**parsed, "activated_categories": kept}, ensure_ascii=False))

results["parsed_canonical"] = canonical_column
results["parsed_grounded"] = grounded_column
results["parsed_final"] = final_column
results.to_csv(final_dir / "elmundo_results_oldrun_filtered.csv", index=False, encoding="utf-8-sig")

for name, rows in logs.items():
    pd.DataFrame(rows).to_csv(final_dir / f"elmundo_oldrun_log_{name}.csv",
                              index=False, encoding="utf-8-sig")

In [ ]:
# how many activations survive each step
stages = [("returned", "parsed"), ("after canonicalization", "parsed_canonical"),
          ("after grounding", "parsed_grounded"), ("after role", "parsed_final")]
returned = count_activations(results, "parsed")
for label, column in stages:
    print(f"  activations {label:22s} {count_activations(results, column):7d}")
print(f"  relabeled {len(logs['relabeled'])}")
for name in ["dropped_unknown", "dropped_grounding", "dropped_role"]:
    print(f"    {name:18s} {len(logs[name]):7d} "
          f"({len(logs[name]) / max(returned, 1) * 100:5.2f}% of returned)")

summarize("elmundo (old run)", results)

In [ ]:
# why did each dropped quotation fail: invented, merged with an invented sentence, or stitched
# together from passages that are real but not adjacent
import re

dropped = pd.read_csv(final_dir / "elmundo_oldrun_log_dropped_grounding.csv")
source = read_results_csv(final_dir / "elmundo_final_data_v2.csv")
text_by_id = dict(zip(source["article_url"], source["text"]))

rows = []
for _, r in dropped.dropna(subset=["text_evidence"]).iterrows():
    article = text_by_id.get(r["article_id"], "")
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(r["text_evidence"]))
                 if len(s.strip()) > 15]
    if not sentences:
        continue
    found = sum(is_grounded(s, article) for s in sentences)
    rows.append({"category": r["category"], "found": found, "total": len(sentences)})

stats = pd.DataFrame(rows)
print(f"{len(stats)} dropped quotations examined of {len(dropped)}")
print(f"  nothing found, invented outright      {int((stats['found'] == 0).sum()):6d}")
print(f"  partly found, invented sentence added {int(((stats['found'] > 0) & (stats['found'] < stats['total'])).sum()):6d}")
print(f"  all sentences found but not adjacent  {int((stats['found'] == stats['total']).sum()):6d}")
print(f"  of which single-sentence quotations   {int((stats['total'] == 1).sum()):6d}")
print()
print(stats.groupby("category")["found"].agg(["size", "mean"]).to_string())

### new data with title

In [ ]:
elmundo_results = process_outlet("elmundo")

### Check old and new results

In [ ]:
# compare the old el mundo run with the new one, category shares and the articles that changed most
old = read_results_csv(final_dir / "elmundo_results_oldrun_filtered.csv")
new = read_results_csv(final_dir / "elmundo_results_final.csv")


def category_names(value):
    parsed = parse_row(value)
    return None if parsed is None else frozenset(e["category"] for e in parsed["activated_categories"])


both = old[["article_id", "parsed_final"]].merge(
    new[["article_id", "title", "parsed_final"]], on="article_id", suffixes=("_old", "_new"))
both["old"] = both["parsed_final_old"].apply(category_names)
both["new"] = both["parsed_final_new"].apply(category_names)
both = both[both["old"].notna() & both["new"].notna()].reset_index(drop=True)
both["changed"] = [len(a ^ b) for a, b in zip(both["old"], both["new"])]

shares = pd.DataFrame({
    "old %": pd.Series([c for s in both["old"] for c in s]).value_counts() / len(both) * 100,
    "new %": pd.Series([c for s in both["new"] for c in s]).value_counts() / len(both) * 100,
}).fillna(0).round(1).sort_index()
print(len(both), "articles usable in both runs")
print("without any activation | old", round((both["old"].apply(len) == 0).mean() * 100, 1),
      "% | new", round((both["new"].apply(len) == 0).mean() * 100, 1), "%")
print("identical category sets:", int((both["changed"] == 0).sum()))
print()
print(shares.to_string())

In [ ]:
# print the activations of one article in the old and in the new run, with their evidence
def compare_article(article_id):
    row = both[both["article_id"] == article_id].iloc[0]
    print("=" * 100)
    print(row["title"])
    print(article_id)
    for label, column in [("old run", "parsed_final_old"), ("new run", "parsed_final_new")]:
        print(f"  {label}")
        for e in parse_row(row[column])["activated_categories"]:
            print(f"    {e['category']} | russia as {e.get('russia_role')}")
            if e.get("text_evidence"):
                print(f"        text: {e['text_evidence']}")
            for image in e.get("image_evidence") or []:
                print(f"        image: {image.get('image_file')} | {image.get('reason')}")


# the ten articles whose category sets changed most between the two runs
for article_id in both.sort_values("changed", ascending=False)["article_id"].head(10):
    compare_article(article_id)

In [ ]:
# count el mundo live blogs left in the corpus and show the context around phrases that look like reader comments
df = read_results_csv(final_dir / "elmundo_final_data_v2.csv")
print("live blogs (-directo.html):", int(df["article_url"].str.contains(r"-directo\.html", regex=True).sum()))
print()

for phrase in ["absurda tiranía rusa", "envenenar a Biden", "Es un asesino de masas"]:
    for _, row in df[df["text"].str.contains(phrase, regex=False)].iterrows():
        i = row["text"].find(phrase)
        print(row["article_url"])
        print(repr(row["text"][max(0, i - 600): i + 300]))
        print("-" * 100)

### Cut comments from the posts 

In [ ]:
# keep the original el mundo file and define how the comment section is recognised
import shutil

original_path = final_dir / "elmundo_final_data_v2_with_comments.csv"
data_path = final_dir / "elmundo_final_data_v2.csv"
if not original_path.exists():
    shutil.copy(data_path, original_path)
original = read_results_csv(original_path)

marker = "Conforme a los criterios de The Trust Project"
director_line = "El director de El Mundo selecciona las noticias de mayor interés para ti."
empty_comment_box = ("Todavía no hay comentarios", "No admite comentarios")


# return the article body without the comment section and the rule that decided the cut
def remove_comments(url, text, article_last_paragraph=frozenset()):
    paragraphs = text.split("\n")
    if "-directo" in url:
        return text, "live blog, left as is"
    if "-video" in url:
        return text, "video page, no comment section"
    for i, p in enumerate(paragraphs):
        if p.startswith(marker):
            return "\n".join(paragraphs[:i]).rstrip(), "cut at the trust project block"
    if director_line in paragraphs:
        return "\n".join(paragraphs[:paragraphs.index(director_line)]).rstrip(), "cut at the director line"
    if any(s in paragraphs[-1] for s in empty_comment_box):
        return "\n".join(paragraphs[:-1]).rstrip(), "empty comment box removed"
    if url in article_last_paragraph:
        return text, "last paragraph kept, it belongs to the article"
    return "\n".join(paragraphs[:-1]).rstrip(), "last paragraph removed as a comment"


candidates = original[[remove_comments(u, t)[1] == "last paragraph removed as a comment"
                       for u, t in zip(original["article_url"], original["text"])]]
print("articles whose last paragraph has to be checked on the page:", len(candidates))

In [ ]:
# fetch each of those pages and find whether the last paragraph sits in the article body or in the comment block
import re
import time
import random
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
           "Accept-Language": "es-ES,es;q=0.9"}


def squeeze(text):
    return re.sub(r"\s+", "", str(text))


def locate_last_paragraph(url, last_paragraph):
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    key = squeeze(last_paragraph)[:60]
    for element in soup.find_all(["p", "div", "span", "li"]):
        if key and squeeze(element.get_text(" ", strip=True)).startswith(key):
            chain = [element] + [a for a in element.parents if a.name not in ("[document]", "html", "body")]
            path = [f'{a.name}.{".".join(a.get("class") or [])}#{a.get("id") or ""}' for a in chain]
            in_comments = any(re.search(r"coment|comment", step, re.I) for step in path)
            return ("comment" if in_comments else "article"), " < ".join(path[:6])
    return "not found", ""


check_rows = []
for i, (_, row) in enumerate(candidates.iterrows(), start=1):
    last_paragraph = row["text"].split("\n")[-1]
    try:
        verdict, path = locate_last_paragraph(row["article_url"], last_paragraph)
    except Exception as exc:
        verdict, path = "error", str(exc)
    check_rows.append({"article_url": row["article_url"], "verdict": verdict, "path": path,
                       "last_paragraph": last_paragraph})
    if i % 50 == 0:
        print(f"  {i}/{len(candidates)}")
    time.sleep(random.uniform(0.5, 1.5))

last_paragraph_check = pd.DataFrame(check_rows)
last_paragraph_check.to_csv(final_dir / "elmundo_last_paragraph_check.csv", index=False, encoding="utf-8-sig")
print(last_paragraph_check["verdict"].value_counts().to_string())
print()
for verdict in ["comment", "article", "not found", "error"]:
    for _, r in last_paragraph_check[last_paragraph_check["verdict"] == verdict].head(3).iterrows():
        print(verdict, "|", r["path"][:160])
        print("   ", r["last_paragraph"][:150])

In [ ]:
# remove the comment section, keeping every last paragraph the page shows inside the article body
last_paragraph_check = read_results_csv(final_dir / "elmundo_last_paragraph_check.csv")
article_last_paragraph = frozenset(last_paragraph_check.loc[last_paragraph_check["verdict"] == "article", "article_url"])

clean = original.copy()
clean["text"], clean["cleaning_rule"] = zip(*[remove_comments(u, t, article_last_paragraph)
                                              for u, t in zip(original["article_url"], original["text"])])
clean["removed_text"] = [t[len(c):].strip() for t, c in zip(original["text"], clean["text"])]
clean.to_csv(data_path, index=False, encoding="utf-8-sig")

live_blog_urls = set(clean.loc[clean["article_url"].str.contains("-directo", regex=False), "article_url"])
print(clean["cleaning_rule"].value_counts().to_string())
print()
print("articles with text removed:", int((clean["removed_text"] != "").sum()))
print("shorter than 268 chars after cleaning, live blogs aside:",
      int(((clean["text"].str.len() < 268) & ~clean["article_url"].isin(live_blog_urls)).sum()))

In [ ]:
# find the activations whose text evidence lies only in the removed comments
checkpoint = read_results_csv(final_dir / "elmundo_checkpoint.csv")
before_doc = {u: coded_document(t, x) for u, t, x in zip(original["article_url"], original["title"], original["text"])}
after_doc = {u: coded_document(t, x) for u, t, x in zip(clean["article_url"], clean["title"], clean["text"])}
rule_by_url = dict(zip(clean["article_url"], clean["cleaning_rule"]))

rows = []
for _, row in checkpoint.iterrows():
    parsed = parse_row(row["parsed"])
    if not isinstance(parsed, dict):
        continue
    for e in parsed["activated_categories"]:
        evidence = e.get("text_evidence")
        if not evidence:
            continue
        rows.append({"article_id": row["article_id"],
                     "cleaning_rule": rule_by_url.get(row["article_id"]),
                     "category": canonical_label(e.get("category"), e.get("orientation")) or e.get("category"),
                     "has_image": bool(e.get("image_evidence")),
                     "found_before": is_grounded(evidence, before_doc.get(row["article_id"], "")),
                     "found_after": is_grounded(evidence, after_doc.get(row["article_id"], "")),
                     "text_evidence": evidence})
evidence_check = pd.DataFrame(rows)
from_comments = evidence_check[evidence_check["found_before"] & ~evidence_check["found_after"]]

print("text activations:", len(evidence_check))
print("quoting the removed comments:", len(from_comments), "in", from_comments["article_id"].nunique(), "articles",
      "| of them kept as image activations:", int(from_comments["has_image"].sum()))
print()
print(from_comments["cleaning_rule"].value_counts().to_string())
print()
print(from_comments["category"].value_counts().sort_index().to_string())
print()
for _, r in from_comments.sample(min(15, len(from_comments)), random_state=1).iterrows():
    print(f'{r["category"]} | {r["text_evidence"][:140]}')

In [ ]:
# compare el mundo before and after removing the comments, reapplying the filters without sending anything to the model
def category_shares(results, exclude):
    parsed = results[~results["article_id"].isin(exclude)]["parsed_final"].apply(parse_row).dropna()
    sets = parsed.apply(lambda p: {e["category"] for e in p["activated_categories"]})
    shares = pd.Series([c for s in sets for c in s], dtype=object).value_counts() / len(sets) * 100
    return shares, len(sets), (sets.apply(len) == 0).mean() * 100


source_df = read_results_csv(final_dir / "elmundo_final_data_v2.csv")
live_blog_urls = set(source_df.loc[source_df["article_url"].str.contains("-directo", regex=False), "article_url"])

before_shares, before_n, before_empty = category_shares(
    read_results_csv(final_dir / "elmundo_results_final.csv"), live_blog_urls)

results_df = read_results_csv(final_dir / "elmundo_checkpoint.csv")
elmundo_results, logs = apply_filters(results_df, source_df)
elmundo_results.to_csv(final_dir / "elmundo_results_final.csv", index=False, encoding="utf-8-sig")
for name, log_rows in logs.items():
    pd.DataFrame(log_rows).to_csv(final_dir / f"elmundo_log_{name}.csv", index=False, encoding="utf-8-sig")
report_filter_shares(elmundo_results, logs)
summarize("elmundo", elmundo_results)

after_shares, after_n, after_empty = category_shares(elmundo_results, live_blog_urls)
comparison = pd.DataFrame({"before %": before_shares, "after %": after_shares}).fillna(0)
comparison["difference"] = comparison["after %"] - comparison["before %"]
print()
print(f"articles, live blogs left out | before {before_n} | after {after_n}")
print(f"without any activation | before {before_empty:.1f}% | after {after_empty:.1f}%")
print()
print(comparison.round(1).sort_index().to_string())

### No live-blog 

In [ ]:
# category shares for el mundo without live blogs, counting each category once per article, after all three filters
results = read_results_csv(final_dir / "elmundo_results_final.csv")
results = results[~results["article_id"].str.contains("-directo", regex=False)]

parsed = results["parsed_final"].apply(parse_row)
print("articles without live blogs:", len(results))
print("of them with an unusable answer:", int(parsed.isna().sum()))
parsed = parsed.dropna()
print("articles counted:", len(parsed))
print()

category_sets = parsed.apply(lambda p: {e["category"] for e in p["activated_categories"]})
table = pd.DataFrame({"category": sorted(required_role)})
table["articles"] = [int(category_sets.apply(lambda s: c in s).sum()) for c in table["category"]]
table["share %"] = (table["articles"] / len(parsed) * 100).round(1)
print(table.to_string(index=False))
print()
empty = int((category_sets.apply(len) == 0).sum())
print(f"without any activation: {empty} ({empty / len(parsed) * 100:.1f}%)")

In [ ]:
# category shares for el mundo with live blogs, without them, and for live blogs alone
results = read_results_csv(final_dir / "elmundo_results_final.csv")
results["live_blog"] = results["article_id"].str.contains("-directo", regex=False)
results["parsed_row"] = results["parsed_final"].apply(parse_row)
results = results[results["parsed_row"].notna()]
results["categories"] = results["parsed_row"].apply(lambda p: {e["category"] for e in p["activated_categories"]})


def shares(part):
    column = [round(part["categories"].apply(lambda s: c in s).mean() * 100, 1) for c in sorted(required_role)]
    return column + [round((part["categories"].apply(len) == 0).mean() * 100, 1), len(part)]


comparison = pd.DataFrame({
    "with live blogs %": shares(results),
    "without live blogs %": shares(results[~results["live_blog"]]),
    "live blogs only %": shares(results[results["live_blog"]]),
}, index=sorted(required_role) + ["without any activation", "articles"])
print(comparison.to_string())

In [ ]:
# drop the el mundo live blogs from the data file and rebuild the results and logs without them, without calling the model
source_df = read_results_csv(final_dir / "elmundo_final_data_v2.csv")
live_blog = source_df["article_url"].str.contains("-directo", regex=False)
print("articles before:", len(source_df), "| live blogs removed:", int(live_blog.sum()))
source_df = source_df[~live_blog]
source_df.to_csv(final_dir / "elmundo_final_data_v2.csv", index=False, encoding="utf-8-sig")
print("articles after:", len(source_df))
print()

results_df = read_results_csv(final_dir / "elmundo_checkpoint.csv")
results_df = results_df[results_df["article_id"].isin(set(source_df["article_url"]))]
elmundo_results, logs = apply_filters(results_df, source_df)
elmundo_results.to_csv(final_dir / "elmundo_results_final.csv", index=False, encoding="utf-8-sig")
for name, log_rows in logs.items():
    pd.DataFrame(log_rows).to_csv(final_dir / f"elmundo_log_{name}.csv", index=False, encoding="utf-8-sig")
report_filter_shares(elmundo_results, logs)
summarize("elmundo", elmundo_results)

## Poland

### Polityka

In [ ]:
polityka_results = process_outlet("polityka")

### Rzeczpospolita

In [ ]:
rp_results = process_outlet("rp")

In [ ]:
rp_results = process_outlet("rp")

In [ ]:
# how many articles have no activation at each filter stage
for column in ["parsed", "parsed_canonical", "parsed_grounded", "parsed_final"]:
    rows = rp_results[column].apply(parse_row)
    coded = rows.dropna()
    empty = int(coded.apply(lambda p: len(p.get("activated_categories") or []) == 0).sum())
    print(f"{column:18s} без активаций {empty:6d} из {len(coded)} ({empty / len(coded) * 100:5.1f}%)")

## Hungary

### Telex

In [ ]:
telex_results = process_outlet("telex")

### Magyar Nemzet

In [ ]:
magyar_results = process_outlet("magyar")

# Final prompt stability test

In [ ]:
# code 25 articles per country three times with the final prompt and save every run
stability_n = 25
stability_runs = 3
country_outlets = {"Germany": ["spiegel", "welt"], "Spain": ["eldiario", "elmundo"],
                   "Poland": ["polityka", "rp"], "Hungary": ["telex", "magyar"]}

frames = []
for country, outlets in country_outlets.items():
    for source in outlets:
        df = read_results_csv(final_dir / f"{source}_final_data_v2.csv")
        df["images_folder"] = str(final_dir / f"{source}_final_images")
        df["outlet"] = source
        df["country"] = country
        frames.append(df)
stability_sample = pd.concat(frames, ignore_index=True).groupby("country").sample(n=stability_n, random_state=42)
stability_sample[["article_url", "outlet", "country", "title", "text"]].to_csv(
    final_dir / "stability_sample.csv", index=False, encoding="utf-8-sig")


def code_once(row, run):
    result = code_article(row)
    return {"article_id": row["article_url"], "outlet": row["outlet"], "country": row["country"],
            "run": run,
            "parsed": json.dumps(result["parsed"], ensure_ascii=False) if result["parsed"] else "",
            "cost": result["cost"]}


stability_rows = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(code_once, row, run)
               for _, row in stability_sample.iterrows() for run in range(1, stability_runs + 1)]
    for future in as_completed(futures):
        try:
            stability_rows.append(future.result())
        except Exception as exc:
            print("failed:", exc)
        if len(stability_rows) % 50 == 0:
            print(f"  {len(stability_rows)}/{len(futures)}")

stability = pd.DataFrame(stability_rows)
stability.to_csv(final_dir / "stability_runs.csv", index=False, encoding="utf-8-sig")
print("calls:", len(stability), "| cost $", round(pd.to_numeric(stability["cost"], errors="coerce").sum(), 2))

In [ ]:
# apply the three filters to every run and compare the category sets the three runs return
stability = read_results_csv(final_dir / "stability_runs.csv")
stability_sample = read_results_csv(final_dir / "stability_sample.csv")
filtered, _ = apply_filters(stability, stability_sample)


def category_set(value):
    parsed = parse_row(value)
    return None if parsed is None else frozenset(
        (e["category"], e["orientation"]) for e in parsed["activated_categories"])


def jaccard(a, b):
    return 1.0 if not a and not b else len(a & b) / len(a | b)


filtered["categories"] = filtered["parsed_final"].apply(category_set)
agreement = []
for (article_id, country), group in filtered.groupby(["article_id", "country"]):
    sets = group.sort_values("run")["categories"].tolist()
    if len(sets) < stability_runs or any(s is None for s in sets):
        continue
    pairs = [jaccard(a, b) for i, a in enumerate(sets) for b in sets[i + 1:]]
    agreement.append({"article_id": article_id, "country": country,
                      "jaccard": sum(pairs) / len(pairs),
                      "identical": all(s == sets[0] for s in sets),
                      "any_activation": any(len(s) > 0 for s in sets),
                      "in_any_run": frozenset().union(*sets),
                      "in_all_runs": frozenset.intersection(*sets)})
agreement = pd.DataFrame(agreement)

by_country = agreement.groupby("country").agg(
    complete=("article_id", "size"),
    with_activation=("any_activation", "sum"),
    identical_sets=("identical", "sum"),
    mean_jaccard_all=("jaccard", "mean"))
by_country["mean_jaccard_with_activation"] = agreement[agreement["any_activation"]].groupby("country")["jaccard"].mean()
print(by_country.round(3).to_string())
print()
print("all countries | complete", len(agreement),
      "| identical", int(agreement["identical"].sum()),
      "| mean jaccard", round(agreement["jaccard"].mean(), 3),
      "| with activation", round(agreement.loc[agreement["any_activation"], "jaccard"].mean(), 3))
print()

categories = sorted({c for s in agreement["in_any_run"] for c in s})
per_category = pd.DataFrame([{
    "category": c[0], "orientation": c[1],
    "articles where it appeared": int(agreement["in_any_run"].apply(lambda s: c in s).sum()),
    "held in all runs": int(agreement["in_all_runs"].apply(lambda s: c in s).sum())}
    for c in categories])
print(per_category.to_string(index=False))

## New grounding filter to not drop images 

In [ ]:
# reapply the three filters with the corrected grounding filter to every outlet, without sending anything to the model
for source in ["spiegel", "welt", "eldiario", "elmundo", "polityka", "rp", "telex", "magyar"]:
    print(source)
    source_df = read_results_csv(final_dir / f"{source}_final_data_v2.csv")
    results_df = read_results_csv(final_dir / f"{source}_checkpoint.csv")
    results_df = results_df[results_df["article_id"].isin(set(source_df["article_url"]))]
    filtered_df, logs = apply_filters(results_df, source_df)
    filtered_df.to_csv(final_dir / f"{source}_results_final.csv", index=False, encoding="utf-8-sig")
    for name, log_rows in logs.items():
        pd.DataFrame(log_rows).to_csv(final_dir / f"{source}_log_{name}.csv", index=False, encoding="utf-8-sig")
    report_filter_shares(filtered_df, logs)
    summarize(source, filtered_df)
    print()